In [1]:
# =========================================================
# 02. PREPROCESSING & E-COMMERCE FEATURE ENGINEERING PIPELINE
# =========================================================

import os
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Ensure destination directories exist
os.makedirs('models', exist_ok=True)
os.makedirs('data', exist_ok=True)

# ---------------------------------------------------------
# 1. LOAD DATASET WITH FLEXIBLE PATH RESOLUTION
# ---------------------------------------------------------
csv_filename = 'data_ecommerce_customer_churn.csv'

candidate_paths = [
    os.path.join('data', csv_filename),
    csv_filename,
    os.path.join('..', 'data', csv_filename),
    os.path.join('..', csv_filename)
]

csv_path = next((p for p in candidate_paths if os.path.exists(p)), None)

if csv_path is None:
    raise FileNotFoundError(f"Could not locate {csv_filename} in any candidate path: {candidate_paths}")

df = pd.read_csv(csv_path)

if 'CustomerID' in df.columns:
    df = df.drop(columns=['CustomerID'])

required_columns = [
    'Tenure', 'WarehouseToHome', 'NumberOfDeviceRegistered',
    'PreferedOrderCat', 'SatisfactionScore', 'MaritalStatus',
    'NumberOfAddress', 'Complain', 'DaySinceLastOrder',
    'CashbackAmount', 'Churn'
]

df = df[[c for c in required_columns if c in df.columns]]

X = df.drop(columns=['Churn'])
y = df['Churn']

# ---------------------------------------------------------
# 2. SPLIT TRAIN / TEST BEFORE FITTING
# ---------------------------------------------------------
RANDOM_STATE = 42

X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.20, 
    random_state=RANDOM_STATE, 
    stratify=y
)

print(f"Dataset successfully loaded from: {csv_path}")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")

# ---------------------------------------------------------
# 3. REPRODUCIBLE E-COMMERCE FEATURE ENGINEERING TRANSFORMER
# ---------------------------------------------------------
class EcommerceFeatureEngineer(BaseEstimator, TransformerMixin):
    """
    Engineers e-commerce domain features from raw input columns.
    Ensures identical feature creation during model training and Streamlit inference.
    """
    def __init__(self):
        pass

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        
        if 'PreferedOrderCat' in X.columns:
            X['PreferedOrderCat'] = X['PreferedOrderCat'].replace({'Mobile': 'Mobile Phone'})

        # 1. Cashback Earned per Month of Tenure
        tenure_safe = X['Tenure'].fillna(X['Tenure'].median())
        X['CashbackPerTenure'] = X['CashbackAmount'] / (tenure_safe + 1.0)

        # 2. Inactivity Ratio
        days_since_order_safe = X['DaySinceLastOrder'].fillna(X['DaySinceLastOrder'].median())
        X['InactivityRatio'] = days_since_order_safe / (tenure_safe * 30.0 + 1.0)

        # 3. High Risk Complain Flag
        X['HighRiskComplain'] = (
            (X['Complain'] == 1) & (X['SatisfactionScore'] <= 2)
        ).astype(int)

        # 4. Tenure Lifecycle Stage
        X['TenureStage'] = pd.cut(
            tenure_safe,
            bins=[-np.inf, 3, 12, 24, np.inf],
            labels=['Onboarding', 'Early', 'Established', 'Loyal']
        ).astype(str)

        return X

# ---------------------------------------------------------
# 4. BUILD SKLEARN COLUMNTRANSFORMER PIPELINE
# ---------------------------------------------------------
num_features = [
    'Tenure', 'WarehouseToHome', 'NumberOfDeviceRegistered',
    'SatisfactionScore', 'NumberOfAddress', 'Complain',
    'DaySinceLastOrder', 'CashbackAmount', 'CashbackPerTenure',
    'InactivityRatio', 'HighRiskComplain'
]

cat_features = [
    'PreferedOrderCat', 'MaritalStatus', 'TenureStage'
]

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor_transformer = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, num_features),
        ('cat', cat_pipeline, cat_features)
    ]
)

full_pipeline = Pipeline([
    ('feature_engineer', EcommerceFeatureEngineer()),
    ('preprocessor', preprocessor_transformer)
])

# ---------------------------------------------------------
# 5. FIT PIPELINE ON TRAIN SET & TRANSFORM
# ---------------------------------------------------------
X_train_processed = full_pipeline.fit_transform(X_train)
X_test_processed = full_pipeline.transform(X_test)

print(f"\nFinal Processed Train Features Shape: {X_train_processed.shape}")
print(f"Final Processed Test Features Shape:  {X_test_processed.shape}")

# ---------------------------------------------------------
# 6. SAVE PIPELINE & PROCESSED ARTIFACTS
# ---------------------------------------------------------
joblib.dump(full_pipeline, 'models/preprocessor.joblib')

np.savez_compressed(
    'data/ecom_processed_data.npz',
    X_train=X_train_processed,
    X_test=X_test_processed,
    y_train=y_train.values,
    y_test=y_test.values
)

print("Saved artifacts successfully:")
print(" - models/preprocessor.joblib")
print(" - data/ecom_processed_data.npz")

Dataset successfully loaded from: ..\data\data_ecommerce_customer_churn.csv
X_train shape: (3152, 10)
X_test shape:  (789, 10)

Final Processed Train Features Shape: (3152, 23)
Final Processed Test Features Shape:  (789, 23)
Saved artifacts successfully:
 - models/preprocessor.joblib
 - data/ecom_processed_data.npz
